# Nepal Maize Drought Q1 -- Full Automated Pipeline

**This notebook works in both Google Colab and local Jupyter/VS Code** -- it auto-detects which environment you're in and adjusts file upload and path handling accordingly. No hardcoded usernames or paths.

**Run every cell top to bottom, in order.** Each cell prints a clear PASS/FAIL/warning so you always know what to do next.

What this notebook does:
1. Detects Colab vs local and sets up a safe working folder
2. Installs all required packages
3. Lets you upload/select `Nepal_Maize_Drought_Q1.zip` and verifies it's the latest script versions
4. Walks you through credentials for CDS (ERA5-Land), NASA Earthdata (SMAP), and Google Earth Engine (Sentinel-1/2)
5. Runs the full download pipeline (scripts 01-09), continuing past individual failures and reporting a clear summary
6. Packages the results for you to download or find locally

## 1. Environment detection and safe working folder

In [ ]:
import os, sys, platform

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

IS_COLAB = in_colab()

if IS_COLAB:
    BASE_DIR = "/content"
else:
    # Dedicated subfolder under the user's home directory -- NEVER write
    # directly into the home directory root (Windows blocks this with
    # PermissionError, and it's poor practice on any OS).
    BASE_DIR = os.path.join(os.path.expanduser("~"), "NepalMaizeDroughtWork")

os.makedirs(BASE_DIR, exist_ok=True)
PROJECT_ROOT = os.path.join(BASE_DIR, "Nepal_Maize_Drought_Q1")
CODE_DIR = os.path.join(PROJECT_ROOT, "07_code")

print(f"Environment: {'Google Colab' if IS_COLAB else 'Local (' + platform.system() + ')'}")
print(f"Python: {sys.version}")
print(f"Base working directory: {BASE_DIR}")
print(f"Project will be at: {PROJECT_ROOT}")

## 2. Install dependencies

In [ ]:
!"{sys.executable}" -m pip install -q cdsapi earthaccess earthengine-api geopandas shapely rasterio xarray netCDF4 rioxarray climate-indices scikit-learn xgboost lightgbm shap statsmodels matplotlib pandas requests
print("Dependencies installed.")

## 3. Upload/locate the project zip and extract it
This works whether you're in Colab (browser upload) or local Jupyter/VS Code (native file picker).

In [ ]:
import zipfile, shutil

if os.path.isdir(PROJECT_ROOT):
    print(f"{PROJECT_ROOT} already exists -- deleting so we get a fresh copy with the latest scripts.")
    shutil.rmtree(PROJECT_ROOT)

zip_path = None

if IS_COLAB:
    from google.colab import files
    print("Select Nepal_Maize_Drought_Q1.zip from your computer:")
    uploaded = files.upload()
    zip_path = list(uploaded.keys())[0]
else:
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        zip_path = filedialog.askopenfilename(
            title="Select Nepal_Maize_Drought_Q1.zip",
            initialdir=os.path.join(os.path.expanduser("~"), "Downloads"),
            filetypes=[("Zip files", "*.zip")],
        )
        root.destroy()
    except Exception as e:
        print(f"Native file picker unavailable ({e}).")

    if not zip_path:
        zip_path = input(
            "Enter the FULL path to Nepal_Maize_Drought_Q1.zip on this machine "
            "(e.g. C:\\Users\\you\\Downloads\\Nepal_Maize_Drought_Q1.zip): "
        ).strip().strip('"')

if not zip_path or not os.path.isfile(zip_path):
    raise RuntimeError(f"No valid zip file selected/found at: {zip_path!r}. Re-run this cell.")

print(f"Using zip: {zip_path}")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(BASE_DIR)

if not os.path.isdir(PROJECT_ROOT):
    candidates = [d for d in os.listdir(BASE_DIR)
                  if os.path.isdir(os.path.join(BASE_DIR, d))
                  and "Nepal_Maize_Drought" in d
                  and d not in ("sample_data",)]
    if len(candidates) == 1:
        shutil.move(os.path.join(BASE_DIR, candidates[0]), PROJECT_ROOT)

if not os.path.isdir(PROJECT_ROOT):
    raise RuntimeError(
        f"Extraction did not produce {PROJECT_ROOT}. Check the zip's internal folder "
        f"structure -- contents of {BASE_DIR}: {os.listdir(BASE_DIR)}"
    )

print(f"Project folder ready at {PROJECT_ROOT}")
os.environ["NEPAL_DROUGHT_CODE_DIR"] = CODE_DIR
os.chdir(CODE_DIR)
print("Now in:", os.getcwd())
for entry in sorted(os.listdir(CODE_DIR)):
    print(" ", entry)

In [ ]:
# Verify the ERA5-Land script is the latest (v3, monthly-chunked) version --
# this is the single most common source of confusing errors in this pipeline.
era5_script = os.path.join(CODE_DIR, "02_download_era5_land.py")
with open(era5_script) as f:
    era5_content = f.read()

if "MONTHLY_DIR" in era5_content:
    print("\u2705 Confirmed: latest v3 (monthly-chunked) ERA5-Land script is in place.")
else:
    raise RuntimeError(
        "\u26a0\ufe0f  This is an OLDER version of 02_download_era5_land.py (no monthly "
        "chunking) -- it WILL hit the 'cost limits exceeded' error. Re-download the "
        "latest Nepal_Maize_Drought_Q1.zip from the chat and re-run the upload cell above."
    )

## 4. Setup check (what's configured, what isn't -- nothing downloads yet)

In [ ]:
!"{sys.executable}" 00_setup_check.py

## 5. Credentials (run each cell once per session)

- **CDS (ERA5-Land)**: get/regenerate your key at https://cds.climate.copernicus.eu/profile.
  **Also required once**: accept the license at
  https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land (scroll down, click Accept) --
  every download fails without this, regardless of key validity.
- **NASA Earthdata (SMAP)**: register free at https://urs.earthdata.nasa.gov
- **Google Earth Engine (Sentinel-1/2)**: create a Cloud project at
  https://console.cloud.google.com/projectcreate, then register it (free, non-commercial)
  at https://code.earthengine.google.com/register

In [ ]:
# 5a. Copernicus CDS API key
import getpass

cdsapirc_path = os.path.join(os.path.expanduser("~"), ".cdsapirc")
if os.path.exists(cdsapirc_path):
    print(f"{cdsapirc_path} already exists. Delete it first if you need to enter a new key.")
else:
    cds_key = getpass.getpass("Paste your CDS API key (input hidden): ").strip()
    with open(cdsapirc_path, "w") as f:
        f.write(f"url: https://cds.climate.copernicus.eu/api\nkey: {cds_key}\n")
    del cds_key
    print(f"{cdsapirc_path} written.")

print("\nReminder: have you accepted the license at "
      "https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land yet? "
      "If not, do that now in your browser before running the pipeline.")

In [ ]:
# 5b. NASA Earthdata login (for SMAP)
import earthaccess
auth = earthaccess.login(persist=True)  # prompts for username/password interactively
print("Authenticated:", auth.authenticated)
if not auth.authenticated:
    print("Login failed -- re-run this cell and double check your username/password at "
          "https://urs.earthdata.nasa.gov")

In [ ]:
# 5c. Google Earth Engine
import ee

EE_PROJECT = input("Paste your Earth Engine Cloud project ID: ").strip()
os.environ["EE_PROJECT"] = EE_PROJECT

try:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
    print(f"\u2705 Earth Engine initialized with project: {EE_PROJECT}")
except Exception as e:
    print(f"Earth Engine init failed: {e}")
    print("Check: (1) the project ID is correct, (2) you registered it at "
          "https://code.earthengine.google.com/register")

In [ ]:
# Re-check now that credentials are in place
!"{sys.executable}" 00_setup_check.py

## 6. Run the full pipeline (scripts 01 through 09, in order)

This captures and prints EVERY script's full output (stdout + stderr), so errors are never
silently swallowed. It continues past individual script failures and gives you a clear
PASS/FAIL summary at the end. This step can take a long time (ERA5-Land and Sentinel
downloads are large) -- let it run.

In [ ]:
import subprocess

os.environ["NEPAL_DROUGHT_CODE_DIR"] = CODE_DIR
os.chdir(CODE_DIR)

SCRIPTS = [
    "01_get_boundaries.py",
    "02_download_era5_land.py",
    "03_download_chirps.py",       # large download (~1.5GB/year) -- comment out to skip; optional cross-check only
    "04_download_smap.py",
    "05_download_soilgrids.py",
    "06_download_sentinel_gee.py",
    "07_compute_spi_spei.py",       # EXPECTED to be BLOCKED on a fresh run -- see note below
    "08_build_temporal_coverage_matrix.py",
    "09_data_quality_checks.py",
]

results = []
for script in SCRIPTS:
    print("=" * 72)
    print(f"RUNNING {script}")
    print("=" * 72)
    proc = subprocess.run(
        [sys.executable, script],
        env=os.environ.copy(),
        capture_output=True, text=True,
    )
    print(proc.stdout)
    if proc.stderr:
        print("----- STDERR -----")
        print(proc.stderr)
    status = "PASS" if proc.returncode == 0 else f"FAILED (exit {proc.returncode})"
    results.append((script, status))
    print(f"--- {script}: {status} ---\n")

print("\n" + "=" * 72)
print("PIPELINE SUMMARY")
print("=" * 72)
for script, status in results:
    print(f"  [{status:22s}] {script}")
print("\nNOTE: 07_compute_spi_spei.py is EXPECTED to fail/BLOCK on a fresh run -- it needs a\n"
      "separate long-record (1991-2024) precipitation climatology that\n"
      "02_download_era5_land.py does not pull by default (see that script's docstring).\n"
      "This is intentional: it refuses to compute SPI/SPEI against too-short a record\n"
      "rather than producing misleading values.\n"
      "\nIf any OTHER script shows FAILED, scroll up to its STDERR block above for the\n"
      "exact error, and share that text if you need help fixing it.")

## 7. Fill in the manual yield data (the one step that cannot be automated)

No public API exists for official district-year MoALD/CBS maize yield records. Open
`01_raw_data/yield_moald/yield_moald_TEMPLATE.csv` (via the file browser in Colab's left
sidebar, or directly on disk locally) and follow
`07_code/yield_moald_recovery_notes.md` for where to look. This can be done later --
it doesn't block the rest of the pipeline.

In [ ]:
print(f"Yield template location: {os.path.join(PROJECT_ROOT, '01_raw_data', 'yield_moald', 'yield_moald_TEMPLATE.csv')}")
print(f"Instructions:            {os.path.join(CODE_DIR, 'yield_moald_recovery_notes.md')}")

## 8. Package the results

In [ ]:
results_zip_base = os.path.join(BASE_DIR, "Nepal_Maize_Drought_Q1_RESULTS")
if os.path.exists(results_zip_base + ".zip"):
    os.remove(results_zip_base + ".zip")
shutil.make_archive(results_zip_base, "zip", PROJECT_ROOT)
results_zip = results_zip_base + ".zip"
print(f"Saved: {results_zip}")

if IS_COLAB:
    from google.colab import files as colab_files
    colab_files.download(results_zip)
else:
    print("This file is already on your computer at the path above -- no download step needed.")
    print("Upload it back in the chat when you're ready to continue with the analysis.")